In [2]:
import numpy as np
import pandas as pd

# with open(r"genres.csv", encoding="utf8") as fh:
#     genres = pd.read_csv(fh)
#
#
# print(genres.head())

In [3]:
import pandas as pd

with open(r"new_clean_data(in).csv", encoding="utf8") as fh:
    movies = pd.read_csv(fh)

# movies = movies[["id","description"]]
# movies["genre"] = genres["genre"]
# movies.dropna(subset=["description","genre"], inplace=True)
#
# print(movies.head())
# print(movies["genre"].unique())
# print(movies.shape)
print(movies['description'])

0         Barbie and Ken are having the time of their li...
1         All unemployed, Ki-taek's family takes peculia...
2         An aging Chinese immigrant is swept up in an i...
3         A ticking-time-bomb insomniac and a slippery s...
4         Mia, an aspiring actress, serves lattes to mov...
                                ...                        
397475    A trendy city girl receives an old necklace in...
397476    Fecal Microbiota Transplant (FMT) is the act o...
397477    Athens. 399. Two lovers: Lamprocle, a young ma...
397478    "Khaitseli" tells the story of two siblings fr...
397479    Photographer Diane Arbus believes she is no lo...
Name: description, Length: 397480, dtype: object


In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(max_features=1000, stop_words='english')
X = vectorizer.fit_transform(movies["description"]).todense()
Y = movies["genre"]
print(vectorizer.get_feature_names_out())

['000' '10' '11' '12' '15' '16' '20' '30' 'abandoned' 'able' 'accident'
 'accidentally' 'accused' 'act' 'action' 'actor' 'actors' 'actress'
 'actually' 'adaptation' 'adventure' 'adventures' 'affair' 'africa'
 'african' 'age' 'aged' 'agent' 'ago' 'air' 'album' 'alex' 'alien' 'alive'
 'america' 'american' 'ancient' 'angeles' 'animals' 'animated' 'animation'
 'anna' 'apart' 'apartment' 'appears' 'area' 'army' 'arrival' 'arrives'
 'art' 'artist' 'artists' 'arts' 'asks' 'attack' 'attempt' 'attempts'
 'attention' 'audience' 'author' 'award' 'away' 'baby' 'bad' 'band' 'bank'
 'bar' 'based' 'battle' 'beach' 'beautiful' 'beauty' 'began' 'begin'
 'beginning' 'begins' 'believe' 'believes' 'beloved' 'berlin' 'best'
 'better' 'big' 'biggest' 'birth' 'birthday' 'black' 'blind' 'blood'
 'blue' 'bodies' 'body' 'bond' 'book' 'border' 'born' 'boss' 'boy'
 'boyfriend' 'boys' 'break' 'breaks' 'bring' 'brings' 'british' 'broken'
 'brother' 'brothers' 'brought' 'brutal' 'build' 'building' 'business'
 'busin

In [5]:
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(X, Y, test_size=0.2, random_state=42)

In [6]:
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np
from sklearn.ensemble import AdaBoostClassifier, RandomForestClassifier
from sklearn import tree, ensemble
from sklearn.model_selection import KFold, GridSearchCV
from sklearn.svm import SVC

# hyper_grid = {
#     'n_estimators': [10, 50, 100, 200, 300, 400, 500],
#     'max_depth': [1, 2, 10, 20, 50, 100, None]
# }
# # 5-fold cv partition
# cv5 = KFold(n_splits = 5, shuffle = True, random_state = 3870)
#
# # Forming the empty trees for each combo of parameters in hyper_grid
# hp_grid_search = GridSearchCV(
#     ensemble.RandomForestRegressor(random_state = 3870),
#     hyper_grid,
#     cv = cv5,
#     scoring = 'r2',
#     n_jobs = -1
# )
#
# hp_grid_search.fit(X = np.asarray(x_train), y = y_train)
#
# print(f'The best choice for the number of trees is {hp_grid_search.best_params_['n_estimators']}')
# print(f'The best choice for max depth is {hp_grid_search.best_params_['max_depth']}')
# print(f'which has an R-squared of: {hp_grid_search.best_score_: .4f}')

#clf = DecisionTreeClassifier(max_depth=5, random_state=42)
clf =  RandomForestClassifier(
        max_depth=25, random_state=42) #max_features should maybe change and n_features
                                        #do a grid search yay (tuning_a_random_forest)
#clf = SVC(kernel='rbf', decision_function_shape='ovr')
clf.fit(np.asarray(x_train), y_train)
y_pred = clf.predict(np.asarray(x_test))
score = clf.score(np.asarray(x_test), y_test)
print(score)
print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

0.08946362081110999
                 precision    recall  f1-score   support

         Action       0.07      0.03      0.04      4373
      Adventure       0.16      0.08      0.10      4279
      Animation       0.06      0.02      0.03      4381
         Comedy       0.07      0.09      0.08      4354
          Crime       0.06      0.03      0.04      4300
    Documentary       0.06      0.02      0.03      4342
          Drama       0.07      0.25      0.11      4291
         Family       0.08      0.06      0.07      4270
        Fantasy       0.20      0.11      0.14      4298
        History       0.23      0.18      0.20      4348
         Horror       0.06      0.03      0.04      4375
          Music       0.06      0.14      0.09      4280
        Mystery       0.20      0.11      0.14      4258
        Romance       0.06      0.03      0.04      4326
Science Fiction       0.19      0.12      0.14      4309
       TV Movie       0.06      0.27      0.10      4243
       Thr

In [9]:
import nltk
nltk.download('punkt_tab')
from gensim.models import Word2Vec


# 1. Apply a tokenization function to each element in the Series
# Each entry goes from 'a sentence string' to ['a', 'list', 'of', 'words']
tokenized_sentences = movies['description'].apply(nltk.word_tokenize) # Use .apply() for Series operations

# 2. The result is a pandas Series of lists, which is an iterable
# and can be directly passed to the Word2Vec model for training
model = Word2Vec(tokenized_sentences, min_count=1, vector_size=100, window=5)

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\lilas\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt_tab.zip.
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word

In [20]:
print([tokenized_sentences])
desc_embed = model.wv[np.asarray(tokenized_sentences)]


# x_train, x_test, y_train, y_test = train_test_split(model, Y, test_size=0.2, random_state=42)
#
# clf =  RandomForestClassifier(
#         max_depth=5, n_estimators=10, max_features=1, random_state=42)
# clf.fit(x_train, y_train)
# y_pred = clf.predict(x_test)
# score = clf.score(x_test, y_test)
# print(score)
# print(classification_report(y_test, y_pred))
# print(confusion_matrix(y_test, y_pred))


[0         [Barbie, and, Ken, are, having, the, time, of,...
1         [All, unemployed, ,, Ki-taek, 's, family, take...
2         [An, aging, Chinese, immigrant, is, swept, up,...
3         [A, ticking-time-bomb, insomniac, and, a, slip...
4         [Mia, ,, an, aspiring, actress, ,, serves, lat...
                                ...                        
941585    [One, day, ,, while, Himuro, (, Yasufu, Motomi...
941586    [Himuro, (, Yasufu, Motomiya, ), starts, a, fi...
941593    [Shinjuku, forest, at, night, ., In, the, sap,...
941594    [The, city, that, never, sleeps, ,, where, ins...
941595    [In, a, world, where, order, has, broken, down...
Name: description, Length: 780785, dtype: object]


TypeError: unhashable type: 'list'